# Contexta Research — 10 Hybrid WSD

Built on SenseForge shared modules. Run sequentially from the repository root or notebooks directory. Missing data is explicitly skipped; no fabricated results.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Source directory:", SRC)
print("Source exists:", SRC.exists())

assert (SRC / "senseforge").is_dir(), (
    "SenseForge package not found. Check project root."
)

import senseforge

print("SenseForge imported successfully!")

Project root: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Source directory: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/src
Source exists: True
SenseForge imported successfully!


In [3]:
import inspect

from senseforge.benchmark import route, calibrate
from senseforge.inference import load_wsd

print("=== ROUTE ===")
print(inspect.getsource(route))

print("\n=== CALIBRATE ===")
print(inspect.getsource(calibrate))

print("\n=== LOAD WSD ===")
print(inspect.getsource(load_wsd))

=== ROUTE ===
def route(row,neural=None,threshold=.75,method='rule'):
    base=predict(row['text'],row['word'],method=method)
    if neural is None:return base
    label,confidence=neural(row)
    if confidence>=threshold:return {**base,'sense_id':label,'method':'hybrid_neural','status':'neural_above_threshold','confidence':confidence}
    return {**base,'method':'hybrid_fallback','status':'neural_below_threshold','confidence':confidence}


=== CALIBRATE ===
def calibrate(validation,neural,thresholds=(.5,.6,.7,.8,.9,.95)):
    if not validation:raise ValueError('Validation gold required for calibration')
    scores=[(evaluate(validation,lambda r:route(r,neural,t))['macro_f1'],t) for t in thresholds]
    return sorted(scores,key=lambda x:(-x[0],x[1]))[0][1]


=== LOAD WSD ===
def load_wsd(checkpoint,model_name='xlm-roberta-base',max_length=160):
    import torch
    from transformers import AutoTokenizer
    from .neural import TargetClassifier
    from .training import device_auto
    

In [5]:
from senseforge.core import INVENTORY
from senseforge.benchmark import route
from senseforge.core import predict

examples = [
    {
        "text": "The bank approved the loan.",
        "word": "bank"
    },
    {
        "text": "The fisherman sat on the bank.",
        "word": "bank"
    },
    {
        "text": "The plant needs sunlight.",
        "word": "plant"
    }
]

for row in examples:
    print("\nINPUT:", row)
    print("RULE PREDICTION:")
    print(predict(
        row["text"],
        row["word"],
        method="rule"
    ))

print("\nINVENTORY SAMPLE:")
print(INVENTORY["bank"])


INPUT: {'text': 'The bank approved the loan.', 'word': 'bank'}
RULE PREDICTION:
{'word': 'bank', 'sense_id': 'bank.1', 'definition': 'financial institution', 'hindi': 'बैंक', 'method': 'rule', 'status': 'lexical_rule', 'evidence': ['loan'], 'alternatives': [{'id': 'bank.1', 'hindi': 'बैंक', 'definition': 'financial institution', 'matched_cues': ['loan']}, {'id': 'bank.2', 'hindi': 'किनारा', 'definition': 'river shore water stream nadi नदी', 'matched_cues': []}], 'confidence': None}

INPUT: {'text': 'The fisherman sat on the bank.', 'word': 'bank'}
RULE PREDICTION:
{'word': 'bank', 'sense_id': 'bank.1', 'definition': 'financial institution', 'hindi': 'बैंक', 'method': 'rule', 'status': 'fallback_ambiguous_or_unmatched', 'evidence': [], 'alternatives': [{'id': 'bank.1', 'hindi': 'बैंक', 'definition': 'financial institution', 'matched_cues': []}, {'id': 'bank.2', 'hindi': 'किनारा', 'definition': 'river shore water stream nadi नदी', 'matched_cues': []}], 'confidence': None}

INPUT: {'text

In [7]:
from pathlib import Path

LORA = ROOT / "artifacts" / "wsd_v2_lora"

paths = {
    "Adapter": LORA / "best_lora_adapter",
    "Classifier head": LORA / "best_classifier_head.pt",
    "Training history": LORA / "training_history.json"
}

for name, path in paths.items():
    print(
        f"{name:20}",
        "FOUND" if path.exists() else "MISSING"
    )

assert all(path.exists() for path in paths.values())

print("\nLoRA artifacts verified.")

Adapter              FOUND
Classifier head      FOUND
Training history     FOUND

LoRA artifacts verified.


In [9]:
import gc
import torch
import torch.nn as nn

from transformers import AutoModel, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "xlm-roberta-base"
DEVICE = torch.device("cpu")
MAX_LENGTH = 64

LORA = ROOT / "artifacts" / "wsd_v2_lora"

SENSES = [
    "bank.1", "bank.2",
    "interest.1", "interest.2",
    "plant.1", "plant.2",
    "point.1", "point.2",
    "right.1", "right.2",
    "table.1", "table.2"
]

class LoRAWSDClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size

        self.dropout = nn.Dropout(0.15)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        target_mask,
        class_mask=None
    ):
        hidden = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        weights = target_mask.unsqueeze(-1)

        pooled = (
            (hidden * weights).sum(dim=1)
            / weights.sum(dim=1).clamp(min=1)
        )

        logits = self.classifier(
            self.dropout(pooled)
        )

        if class_mask is not None:
            logits = logits.masked_fill(
                ~class_mask.bool(), -1e4
            )

        return logits


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

base = AutoModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
)

encoder = PeftModel.from_pretrained(
    base,
    LORA / "best_lora_adapter",
    is_trainable=False
)

model = LoRAWSDClassifier(
    encoder,
    len(SENSES)
)

head = torch.load(
    LORA / "best_classifier_head.pt",
    map_location="cpu",
    weights_only=True
)

model.classifier.load_state_dict(head)
model.to(DEVICE)
model.eval()

print("Best LoRA model loaded.")
print("Device:", DEVICE)
print("Supported senses:", len(SENSES))

Best LoRA model loaded.
Device: cpu
Supported senses: 12


In [11]:
import re

def encode_target(text, word):
    matches = list(re.finditer(
        rf"(?<!\w){re.escape(word)}(?!\w)",
        text,
        flags=re.IGNORECASE
    ))

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one occurrence of '{word}'"
        )

    match = matches[0]

    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_offsets_mapping=True
    )

    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]

    positions = [
        i for i, (a, b) in enumerate(offsets)
        if b > a and a < match.end() and b > match.start()
    ]

    if not positions:
        raise ValueError("Target alignment failed")

    capacity = MAX_LENGTH - 2
    first, last = positions[0], positions[-1]

    if last - first + 1 > capacity:
        raise ValueError("Target exceeds context window")

    left = max(1, first - capacity // 2)
    left = min(
        left,
        max(1, last - capacity + 1)
    )
    right = min(len(ids) - 1, left + capacity)

    if right - left < capacity:
        left = max(1, right - capacity)

    selected = list(range(left, right))

    input_ids = (
        [ids[0]]
        + [ids[i] for i in selected]
        + [ids[-1]]
    )

    target_mask = (
        [0.0]
        + [float(i in positions) for i in selected]
        + [0.0]
    )

    attention = [1] * len(input_ids)
    pad = MAX_LENGTH - len(input_ids)

    assert pad >= 0
    assert sum(target_mask) > 0

    input_ids += [tokenizer.pad_token_id] * pad
    attention += [0] * pad
    target_mask += [0.0] * pad

    allowed = [
        sense.startswith(word + ".")
        for sense in SENSES
    ]

    assert sum(allowed) == 2

    return {
        "input_ids": torch.tensor(
            [input_ids], dtype=torch.long, device=DEVICE
        ),
        "attention_mask": torch.tensor(
            [attention], dtype=torch.long, device=DEVICE
        ),
        "target_mask": torch.tensor(
            [target_mask], dtype=torch.float32, device=DEVICE
        ),
        "class_mask": torch.tensor(
            [allowed], dtype=torch.bool, device=DEVICE
        )
    }


@torch.inference_mode()
def lora_infer(row):
    word = row["word"].lower()
    batch = encode_target(row["text"], word)

    logits = model(**batch)
    probs = torch.softmax(logits, dim=-1)[0]

    confidence, index = probs.max(dim=0)

    return (
        SENSES[index.item()],
        float(confidence.item())
    )

print("LoRA inference adapter ready.")

LoRA inference adapter ready.


In [13]:
from senseforge.core import INVENTORY, predict

SUPPORTED_WORDS = {
    sense.split(".")[0]
    for sense in SENSES
}

def hybrid_predict(text, word, threshold=0.75):
    word = word.lower().strip()

    base = predict(
        text,
        word,
        method="rule"
    )

    if word not in SUPPORTED_WORDS:
        return {
            **base,
            "method": "rule_fallback",
            "status": "neural_unsupported_word"
        }

    label, confidence = lora_infer({
        "text": text,
        "word": word
    })

    if confidence < threshold:
        return {
            **base,
            "method": "hybrid_fallback",
            "status": "neural_below_threshold",
            "confidence": confidence
        }

    sense = next(
        s for s in INVENTORY[word]["senses"]
        if s["id"] == label
    )

    return {
        **base,
        "sense_id": label,
        "definition": sense["definition"],
        "hindi": sense["hindi"],
        "method": "hybrid_lora",
        "status": "neural_above_threshold",
        "confidence": confidence,
        "evidence": [],
        "alternatives": base.get("alternatives", [])
    }

print("Hybrid LoRA router ready.")

Hybrid LoRA router ready.


In [19]:
import re
import torch

def encode_target(text, word, start=None, end=None):
    word = word.lower().strip()

    # Prefer gold-annotated character offsets.
    if start is not None and end is not None:
        start, end = int(start), int(end)

        if not (0 <= start < end <= len(text)):
            raise ValueError("Invalid target offsets")

    else:
        matches = list(re.finditer(
            rf"(?<!\w){re.escape(word)}(?!\w)",
            text,
            flags=re.IGNORECASE
        ))

        if len(matches) != 1:
            raise ValueError(
                f"Expected one unambiguous occurrence "
                f"of '{word}', found {len(matches)}. "
                "Supply target offsets."
            )

        start, end = matches[0].span()

    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_offsets_mapping=True
    )

    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]

    positions = [
        i for i, (a, b) in enumerate(offsets)
        if b > a and a < end and b > start
    ]

    if not positions:
        raise ValueError("Target-token alignment failed")

    capacity = MAX_LENGTH - 2
    first, last = positions[0], positions[-1]

    if last - first + 1 > capacity:
        raise ValueError("Target exceeds context window")

    left = max(1, first - capacity // 2)
    left = min(left, max(1, last - capacity + 1))

    right = min(len(ids) - 1, left + capacity)

    if right - left < capacity:
        left = max(1, right - capacity)

    selected = list(range(left, right))
    target_set = set(positions)

    input_ids = (
        [ids[0]]
        + [ids[i] for i in selected]
        + [ids[-1]]
    )

    target_mask = (
        [0.0]
        + [float(i in target_set) for i in selected]
        + [0.0]
    )

    attention = [1] * len(input_ids)
    pad = MAX_LENGTH - len(input_ids)

    assert pad >= 0
    assert sum(target_mask) > 0

    input_ids += [tokenizer.pad_token_id] * pad
    attention += [0] * pad
    target_mask += [0.0] * pad

    allowed = [
        sense.startswith(word + ".")
        for sense in SENSES
    ]

    if sum(allowed) != 2:
        raise ValueError(f"Unsupported word: {word}")

    return {
        "input_ids": torch.tensor(
            [input_ids], dtype=torch.long, device=DEVICE
        ),
        "attention_mask": torch.tensor(
            [attention], dtype=torch.long, device=DEVICE
        ),
        "target_mask": torch.tensor(
            [target_mask], dtype=torch.float32, device=DEVICE
        ),
        "class_mask": torch.tensor(
            [allowed], dtype=torch.bool, device=DEVICE
        )
    }


@torch.inference_mode()
def lora_infer(row):
    word = row["word"].lower().strip()

    batch = encode_target(
        text=row["text"],
        word=word,
        start=row.get("target_char_start"),
        end=row.get("target_char_end")
    )

    logits = model(**batch)
    probs = torch.softmax(logits, dim=-1)[0]

    confidence, index = probs.max(dim=0)

    return (
        SENSES[index.item()],
        float(confidence.item())
    )

print("Offset-aware LoRA inference ready.")

Offset-aware LoRA inference ready.


In [15]:
examples = [
    ("The bank approved the loan.", "bank"),
    ("The fisherman sat on the bank.", "bank"),
    ("The plant needs sunlight.", "plant"),
    ("The factory plant closed yesterday.", "plant"),
    ("Turn right after the bridge.", "right"),
    ("The answer is right.", "right"),
    ("The table contains several rows.", "table"),
    ("She placed the book on the table.", "table")
]

for text, word in examples:
    result = hybrid_predict(
        text, word, threshold=0.75
    )

    print("\nSentence:", text)
    print("Word:", word)
    print("Sense:", result["sense_id"])
    print("Hindi:", result["hindi"])
    print("Method:", result["method"])
    print("Confidence:", result.get("confidence"))


Sentence: The bank approved the loan.
Word: bank
Sense: bank.1
Hindi: बैंक
Method: hybrid_lora
Confidence: 0.9997685551643372

Sentence: The fisherman sat on the bank.
Word: bank
Sense: bank.2
Hindi: किनारा
Method: hybrid_lora
Confidence: 0.9999221563339233

Sentence: The plant needs sunlight.
Word: plant
Sense: plant.1
Hindi: पौधा
Method: hybrid_fallback
Confidence: 0.5366423726081848

Sentence: The factory plant closed yesterday.
Word: plant
Sense: plant.2
Hindi: कारखाना
Method: hybrid_lora
Confidence: 0.9999693632125854

Sentence: Turn right after the bridge.
Word: right
Sense: right.2
Hindi: दायाँ
Method: hybrid_lora
Confidence: 0.9988631010055542

Sentence: The answer is right.
Word: right
Sense: right.1
Hindi: सही
Method: hybrid_lora
Confidence: 0.9940066337585449

Sentence: The table contains several rows.
Word: table
Sense: table.2
Hindi: तालिका
Method: hybrid_lora
Confidence: 0.9999322891235352

Sentence: She placed the book on the table.
Word: table
Sense: table.1
Hindi: मेज

In [25]:
import re
from senseforge.core import predict

def rule_infer(row):
    text = row["text_en"]
    word = row["target_word"]

    start = int(row["target_char_start"])
    end = int(row["target_char_end"])

    if not (0 <= start < end <= len(text)):
        raise ValueError("Invalid target offsets")

    surface = text[start:end]

    # Try original sentence first.
    try:
        return predict(text, word, method="rule")

    except ValueError as error:
        if "Target word not present" not in str(error):
            raise

    # Replace only the annotated target surface with its lemma.
    normalized_text = (
        text[:start]
        + word
        + text[end:]
    )

    result = predict(
        normalized_text,
        word,
        method="rule"
    )

    return {
        **result,
        "status": "rule_lemma_normalized",
        "target_surface": surface
    }

print("Offset-aware rule fallback ready.")

Offset-aware rule fallback ready.


In [27]:
import json
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

V2 = ROOT / "artifacts" / "wsd_v2"
OUT = ROOT / "artifacts" / "hybrid_v2"
OUT.mkdir(parents=True, exist_ok=True)

with (V2 / "validation.jsonl").open(encoding="utf-8") as f:
    val_rows = [
        json.loads(line)
        for line in f if line.strip()
    ]

assert len(val_rows) == 60

cached = []

for i, row in enumerate(val_rows):
    text = row["text_en"]
    word = row["target_word"]

    neural_label, confidence = lora_infer({
    "text": text,
    "word": word,
    "target_char_start": row["target_char_start"],
    "target_char_end": row["target_char_end"]
})
    rule_result = rule_infer(row)

    cached.append({
        "gold": row["sense_id"],
        "word": word,
        "neural": neural_label,
        "confidence": confidence,
        "rule": rule_result["sense_id"]
    })

    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/{len(val_rows)}")

thresholds = np.arange(0.50, 1.00, 0.05)
results = []

gold = [r["gold"] for r in cached]

for threshold in thresholds:
    predictions = [
        r["neural"]
        if r["confidence"] >= threshold
        else r["rule"]
        for r in cached
    ]

    results.append({
        "threshold": round(float(threshold), 2),
        "accuracy": float(
            accuracy_score(gold, predictions)
        ),
        "macro_f1": float(
            f1_score(
                gold, predictions,
                labels=SENSES,
                average="macro",
                zero_division=0
            )
        ),
        "neural_usage": sum(
            r["confidence"] >= threshold
            for r in cached
        ) / len(cached)
    })

for r in results:
    print(
        f"Threshold={r['threshold']:.2f} | "
        f"Accuracy={r['accuracy']:.4f} | "
        f"Macro-F1={r['macro_f1']:.4f} | "
        f"Neural usage={r['neural_usage']:.1%}"
    )

best = max(
    results,
    key=lambda r: (
        r["macro_f1"],
        r["accuracy"],
        r["neural_usage"]
    )
)

print("\nBest validation threshold:", best)

(OUT / "calibration_results.json").write_text(
    json.dumps({
        "selection_split": "wsd_v2_validation",
        "results": results,
        "selected": best
    }, indent=2),
    encoding="utf-8"
)

Processed 20/60
Processed 40/60
Processed 60/60
Threshold=0.50 | Accuracy=1.0000 | Macro-F1=1.0000 | Neural usage=100.0%
Threshold=0.55 | Accuracy=1.0000 | Macro-F1=1.0000 | Neural usage=100.0%
Threshold=0.60 | Accuracy=1.0000 | Macro-F1=1.0000 | Neural usage=100.0%
Threshold=0.65 | Accuracy=1.0000 | Macro-F1=1.0000 | Neural usage=100.0%
Threshold=0.70 | Accuracy=1.0000 | Macro-F1=1.0000 | Neural usage=100.0%
Threshold=0.75 | Accuracy=1.0000 | Macro-F1=1.0000 | Neural usage=100.0%
Threshold=0.80 | Accuracy=1.0000 | Macro-F1=1.0000 | Neural usage=100.0%
Threshold=0.85 | Accuracy=0.9833 | Macro-F1=0.9074 | Neural usage=98.3%
Threshold=0.90 | Accuracy=0.9833 | Macro-F1=0.9074 | Neural usage=96.7%
Threshold=0.95 | Accuracy=0.9833 | Macro-F1=0.9074 | Neural usage=93.3%

Best validation threshold: {'threshold': 0.5, 'accuracy': 1.0, 'macro_f1': 1.0, 'neural_usage': np.float64(1.0)}


1402

In [29]:
import json
from pathlib import Path

HYBRID_OUT = ROOT / "artifacts" / "hybrid_v2"
HYBRID_OUT.mkdir(parents=True, exist_ok=True)

# All thresholds from 0.50 to 0.80 tied.
# Choose 0.75 as a conservative operational setting,
# not as a statistically calibrated probability.
SELECTED_THRESHOLD = 0.75

config = {
    "version": "hybrid_v2",
    "neural_model": "xlmr_lora_epoch2",
    "supported_words": sorted(SUPPORTED_WORDS),
    "sense_labels": SENSES,
    "threshold": SELECTED_THRESHOLD,
    "threshold_selection": "validation_tied_range_0.50_to_0.80",
    "fallback": "rule_based_then_dictionary_default",
    "confidence_type": "uncalibrated_softmax",
    "validation_accuracy": 1.0,
    "validation_macro_f1": 1.0,
    "validation_examples": 60,
    "limitations": [
        "Only six words supported by LoRA",
        "Fallback not independently validated",
        "Sense inventory mappings require independent review"
    ]
}

path = HYBRID_OUT / "hybrid_config.json"

path.write_text(
    json.dumps(config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Hybrid configuration saved:", path)
print("Threshold:", SELECTED_THRESHOLD)
print("Supported words:", len(SUPPORTED_WORDS))

Hybrid configuration saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/hybrid_v2/hybrid_config.json
Threshold: 0.75
Supported words: 6


In [31]:
tests = [
    ("The bank approved the loan.", "bank"),
    ("The fisherman sat on the bank.", "bank"),
    ("The plant needs sunlight.", "plant"),
    ("Turn right after the bridge.", "right"),
    ("He hit the ball with a bat.", "bat"),
    ("A bat flew out of the cave.", "bat")
]

for text, word in tests:
    result = hybrid_predict(
        text,
        word,
        threshold=SELECTED_THRESHOLD
    )

    assert result["sense_id"] in {
        s["id"] for s in INVENTORY[word]["senses"]
    }

    selected_sense = next(
        s for s in INVENTORY[word]["senses"]
        if s["id"] == result["sense_id"]
    )

    assert result["hindi"] == selected_sense["hindi"]
    assert result["definition"] == selected_sense["definition"]

    print(
        f"{word:8} | "
        f"{result['sense_id']:12} | "
        f"{result['hindi']:12} | "
        f"{result['method']}"
    )

print("\nHybrid smoke test passed.")

bank     | bank.1       | बैंक         | hybrid_lora
bank     | bank.2       | किनारा       | hybrid_lora
plant    | plant.1      | पौधा         | hybrid_fallback
right    | right.2      | दायाँ        | hybrid_lora
bat      | bat.1        | बल्ला        | rule_fallback
bat      | bat.2        | चमगादड़      | rule_fallback

Hybrid smoke test passed.
